In [1]:
#use this to benchmark improvements
import pandas as pd
## -- 12/26/24 -- adding new Ferguson dataset from Marty
from dotenv import load_dotenv
import os
load_dotenv()


import pandas as pd

df_final = pd.read_csv(r'data/ferg_plus_gt_44k_size_updated.csv')
print('len of df: ', len(df_final))
#df.head(5)

len of df:  44000


In [2]:
## load this dataset to qdrant for testing
## note that this collection is a sparse filter

# CONNECT TO QDRANT
import os
import json
import qdrant_client
from qdrant_client.http.models import Filter, FieldCondition, MatchValue,  MatchAny, PointStruct, VectorParams, Distance
from qdrant_client import QdrantClient


ENDLESSFORMS_QDRANT_URL = os.getenv("ENDLESSFORMS_QDRANT_URL")
ENDLESSFORMS_TEST_CLUSTER_KEY = os.getenv("ENDLESSFORMS_TEST_CLUSTER_KEY")

qdrantclient = QdrantClient(
    url=ENDLESSFORMS_QDRANT_URL,
    api_key=ENDLESSFORMS_TEST_CLUSTER_KEY,
    timeout=3000
)

print(qdrantclient)

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_HYBRID'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(name='DENSE_VECTOR_BENCHMARK_25K_DEC17'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15_2'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER

In [3]:
# load embedding model, check size

import dotenv

OPENAIKEY = os.getenv("OPEN_AI_KEY")

from openai import OpenAI
openaiclient = OpenAI(api_key=OPENAIKEY)

response = openaiclient.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-large"
)

v = response.data[0].embedding

print('size of embedding: ', len(v))

size of embedding:  3072


In [4]:
def open_ai_vectorizer(openaiclient, model, text):
    """input text to vectorize, output vector"""
    response = openaiclient.embeddings.create(
    input=text,
    model=model)

    return response.data[0].embedding


In [5]:
# #DELETE OLD COLLECTION
COLLECTION_NAME = "DENSE_VECTOR_FERGUSON_DEC_26_OPENAI_LARGE"

collection_name = COLLECTION_NAME

# Delete the collection
response = qdrantclient.delete_collection(collection_name=collection_name)

# Print the response
print(response)

True


In [6]:

from qdrant_client import QdrantClient, models
from qdrant_client.http.models import VectorParams

COLLECTION_NAME = "DENSE_VECTOR_FERGUSON_DEC_26_OPENAI_LARGE"

# Create the collection
if not qdrantclient.collection_exists(COLLECTION_NAME):
    qdrantclient.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE, on_disk = True)
    )

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_HYBRID'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(name='DENSE_VECTOR_BENCHMARK_25K_DEC17'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15_2'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='SPARSE_VECTOR_COLLECTION_TEST_DEC15'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER

In [7]:
## clean data

from EnhancedDataCleanser import DataCleanser

cleanser = DataCleanser(
        default_uppercase=True,
        measurement_standardization=True,
        fraction_conversion=True
    )

df_clean = cleanser.clean_dataframe(df_final)

In [9]:
print('len of clean data: ', len(df_clean))

len of clean data:  44000


In [8]:
df_clean.columns

Index(['UNNAMED 0', 'DESCRIPTION', 'CATEGORY', 'TYPE', 'PRIMARY_SIZE',
       'REDUCING_SIZE', 'LENGTH', 'MATERIAL_NAME', 'MATERIAL_SPECIFICATION',
       'MATERIAL_GRADE', 'PRESSURE_CLASS', 'PRIMARY_SCHEDULE',
       'REDUCING_SCHEDULE', 'MANUFACTURING_PROCESS', 'END_FINISH',
       'END_CONNECTIONS', 'TRIM', 'MISCELLANEOUS', 'CONFIDENCE'],
      dtype='object')

In [10]:

import numpy as np

# LOAD DATA
# this is for single pointstruct loading
# Initialize vectorizer

embedding_model = "text-embedding-3-large"
bad_data_points = []

"""input dataframe, output PointSturct"""
for i, row in df_clean.iterrows():
    payload = {
        "ID": str(i),
        "DESCRIPTION":str(row['DESCRIPTION']),
        "CATEGORY": str(row['CATEGORY']),
        "TYPE": str(row['TYPE']),
        "PRIMARY_SIZE": str(row['PRIMARY_SIZE']),
        "REDUCING_SIZE": str(row['REDUCING_SIZE']),
        "LENGTH": row['LENGTH'],
        "MATERIAL_NAME": str(row['MATERIAL_NAME']),
        "MATERIAL_SPECIFICATION": row['MATERIAL_SPECIFICATION'],
        "MATERIAL_GRADE": row['MATERIAL_GRADE'],
        "PRESSURE_CLASS": row['PRESSURE_CLASS'],
        "PRIMARY_SCHEDULE": row['PRIMARY_SCHEDULE'],
        "REDUCING_SCHEDULE": row['REDUCING_SCHEDULE'],
        "MANUFACTURING_PROCESS": row['MANUFACTURING_PROCESS'],
        "END_FINISH": row['END_FINISH'],
        "END_CONNECTIONS": row['END_CONNECTIONS'],
        "TRIM": row['TRIM'],
        "MISCELLANEOUS": row['MISCELLANEOUS'],
        "CONFIDENCE": row['CONFIDENCE'],
    }

    desc = row['DESCRIPTION']

    if isinstance(desc,str):

        if desc != np.nan or desc != 'nan':

            #v = get_encoded_embedding(desc,encoder=encoder)
            v = open_ai_vectorizer(openaiclient, embedding_model, desc)

            operation_info = qdrantclient.upsert(
                collection_name=COLLECTION_NAME,
                points=[
                    models.PointStruct(
                        id=i,
                        payload=payload,  # Add any additional payload if necessary
                        vector=v
                    )
                ],
            )

            print(operation_info)

    else:
        print('got a bad data point!')
        bad_data_points.append(payload)

print('all files uploaded!')
print('bad datapoints:', bad_data_points)

operation_id=0 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=4 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=5 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=6 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=7 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=8 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=9 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=10 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=11 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=12 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=13 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=14 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=15 status=<UpdateStatus.COMPLETED: 'completed'>
operation_id=16 status=<UpdateStat

In [8]:
print(len(bad_data_points))

7
